# Featured Event — Candidate Parsing & Drill-Down Debug Notebook

Loads the cached Brave candidates and walks through each pipeline stage so you can see exactly what's being kept, drilled, or dropped — and why.

**Prereq:** run the pipeline at least once with `BRAVE_CACHE_DIR` defaulting on (or set explicitly) so `Featured Event/Code/brave_cache/*.json` exists. This notebook reads from that cache.

In [10]:
import os, sys, json
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

# Locate repo root (the dir that contains NewsletterCreation/) and put
# NewsletterCreation/Code on sys.path BEFORE any project imports.
REPO = Path.cwd().resolve()
while REPO.name and not (REPO / 'NewsletterCreation').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'NewsletterCreation' / 'Code'))

# Now project modules are importable. Use importlib.reload so re-running
# this cell picks up changes to the .py files without restarting the kernel.
import importlib
import aggregator_drilldown
import event_date_filter
importlib.reload(aggregator_drilldown)
importlib.reload(event_date_filter)
from aggregator_drilldown import (
    is_aggregator_url, drill_down_candidate, find_primary_url,
    _hostname, expand_listicle,
)
from event_date_filter import filter_candidates_by_date, upcoming_friday

CACHE_DIR = REPO / 'Featured Event' / 'Code' / 'brave_cache'
print('Repo:        ', REPO)
print('Cache dir:   ', CACHE_DIR)
print('Cache files: ', [p.name for p in CACHE_DIR.glob('*.json')])

Repo:         /Users/couch2coders/Documents/GitHub/newsletters
Cache dir:    /Users/couch2coders/Documents/GitHub/newsletters/Featured Event/Code/brave_cache
Cache files:  ['East_Cobb_Connect_round1.json']


## 1. Pick a cached newsletter round to inspect

In [13]:
NEWSLETTER = 'East_Cobb_Connect'
ROUND      = 1

cache_path = CACHE_DIR / f'{NEWSLETTER}_round{ROUND}.json'
candidates = json.loads(cache_path.read_text(encoding='utf-8'))
print(f'Loaded {len(candidates)} candidates from {cache_path.name}\n')

# Plain-text printout — full title, full URL, no truncation.
for i, c in enumerate(candidates, 1):
    print(f"[{i:2}] host={_hostname(c.get('url',''))}")
    print(f"     title: {c.get('title','')}")
    print(f"     url:   {c.get('url','')}")
    print()

Loaded 28 candidates from East_Cobb_Connect_round1.json

[ 1] host=cobbcounty.gov
     title: Cobb's Calendar Highlights - Upcoming Events in the County | Cobb County Georgia
     url:   https://www.cobbcounty.gov/news/cobbs-calendar-highlights-upcoming-events-county-0

[ 2] host=eastcobbnews.com
     title: Weekend Events: Taste of East Cobb festival; concerts; more - East Cobb News
     url:   https://eastcobbnews.com/weekend-events-taste-of-east-cobb-festival-concerts-more/

[ 3] host=discoveratlanta.com
     title: Cool Things To Do in May in Atlanta | Discover Atlanta
     url:   https://discoveratlanta.com/stories/things-to-do/cool-things-to-do-in-may-in-atlanta/

[ 4] host=travelcobb.org
     title: Home - Cobb Travel & Tourism
     url:   https://travelcobb.org/

[ 5] host=cobbcountycourier.com
     title: Things to do this weekend in Cobb County: Friday May 8 to Sunday, May 10 - Cobb Courier
     url:   https://cobbcountycourier.com/2026/05/things-to-do-this-weekend-in-cobb-co

## 2. Stage 1 — Classify each candidate

The pipeline now classifies every aggregator URL into one of three buckets:

- **LANDING/ARCHIVE** (tag pages, business listings, news landing, bare homepage) → **dropped**
- **LISTICLE** (title/URL contains "things to do", "5 things", "weekend events", etc.) → **expanded** into N constituent events
- **AGGREGATOR** (single-event aggregator article) → **kept** as single candidate

Non-aggregator URLs (venue/organizer pages) are kept untouched.

In [ ]:
import importlib, aggregator_drilldown
importlib.reload(aggregator_drilldown)
from aggregator_drilldown import is_aggregator_url
from urllib.parse import urlparse

# ── Pipeline classification rules (mirror Featured_Event.py) ──
LISTICLE_MARKERS = (
    'things to do', 'things-to-do', '5 things', '10 things',
    'weekend checklist', 'weekend events', 'weekend roundup',
    'weekend guide', 'your weekend', 'events this weekend',
    'events this week', 'events you absolutely need',
    'out and about', 'what to do this', "what's happening",
    'upcoming events', 'calendar of events', 'events calendar',
    'fun things to do', 'guide to events', 'things to do in',
)
NON_LISTICLE_URL_PATTERNS = (
    '/tag/', '/tags/', '/category/', '/categories/',
    '/author/', '/authors/', '/archives/', '/archive/',
    '/business/listing/', '/businesses/',
    '/news/local', '/news/police', '/news/crime',
)

def is_listicle(c):
    blob = f"{c.get('title','')} {c.get('url','')}".lower()
    matched = [m for m in LISTICLE_MARKERS if m in blob]
    return matched

def looks_like_landing_or_archive(url):
    p = urlparse(url)
    path = (p.path or '').lower().rstrip('/')
    if not path:
        return True
    return any(pat in path for pat in NON_LISTICLE_URL_PATTERNS)

# Verdicts:
#   LANDING/ARCHIVE → dropped (no event content of its own)
#   LISTICLE        → expanded into N constituent events
#   AGGREGATOR      → kept as single candidate (single-event aggregator article)
#   normal          → kept as single candidate (venue/organizer URL)
counts = {'LANDING/ARCHIVE': 0, 'LISTICLE': 0, 'AGGREGATOR': 0, 'normal': 0}
for i, c in enumerate(candidates, 1):
    url = c.get('url', '')
    title = c.get('title', '')
    host = _hostname(url)
    is_agg = is_aggregator_url(url)
    matched = is_listicle(c) if is_agg else []
    if is_agg and looks_like_landing_or_archive(url):
        verdict = 'LANDING/ARCHIVE'
    elif is_agg and matched:
        verdict = 'LISTICLE'
    elif is_agg:
        verdict = 'AGGREGATOR'
    else:
        verdict = 'normal'
    counts[verdict] += 1
    print(f"[{i:2}] {verdict:<15} host={host}")
    print(f"     title: {title}")
    print(f"     url:   {url}")
    if matched:
        print(f"     markers: {', '.join(matched)}")
    print()

print('=== Summary ===')
for k, v in counts.items():
    print(f'  {k:<15}: {v}')
print(f'  → Will reach Claude: ~{counts["LISTICLE"]} listicles expanded into many + {counts["AGGREGATOR"]} singles + {counts["normal"]} singles')
print(f'  → Will be dropped:   {counts["LANDING/ARCHIVE"]}')

## 3. Stage 2 — Apply the pipeline's expand / keep / drop logic

Mirrors `Featured_Event.py` exactly:
- Landing/archive URLs are dropped entirely
- Listicles get expanded into individual event URLs (full child events with section context)
- Other aggregators are kept as single candidates
- Non-aggregator URLs pass through

In [ ]:
import importlib, aggregator_drilldown
importlib.reload(aggregator_drilldown)
from aggregator_drilldown import expand_listicle, is_aggregator_url

# Mirrors the pipeline EXACTLY (post-Option-B):
#   1. aggregator + landing/archive → DROP
#   2. ANY URL with listicle markers → EXPAND
#   3. everything else (single-event aggregator article, venue page) → KEEP
pool = []
dropped = []
expanded_total = 0

for c in candidates:
    url = c.get('url', '')
    title = c.get('title', '')
    host = _hostname(url)
    is_agg = is_aggregator_url(url)

    # Step 1
    if is_agg and looks_like_landing_or_archive(url):
        dropped.append(('LANDING/ARCHIVE', c))
        print(f"DROPPED  {url}")
        print(f"  title: {title}")
        print(f"  reason: aggregator landing/archive (no event content)")
        print()
        continue

    # Step 2 — listicle expansion (any host)
    if is_listicle(c):
        expanded = expand_listicle(url, listicle_title=title)
        print(f"LISTICLE  {url}")
        print(f"  title: {title}")
        if not expanded:
            pool.append(c)
            print(f"  ↳ no event links found; keeping original URL")
            print()
            continue
        expanded_total += len(expanded)
        pool.extend(expanded)
        print(f"  ↳ extracted {len(expanded)} event candidates:")
        for i, ev in enumerate(expanded, 1):
            print(f"     [{i:2}] {ev['title']} → {ev['url']}")
        print()
        continue

    # Step 3
    pool.append(c)
    label = 'AGGREGATOR (single)' if is_agg else 'venue/organizer URL'
    print(f"KEPT ({label})  {url}")
    print(f"  title: {title}")
    print()

print('=== Pool after expand/keep/drop ===')
print(f'  Total candidates entering date filter: {len(pool)}')
print(f'  Dropped (landing/archive):             {len(dropped)}')
print(f'  Expanded from listicles:               {expanded_total}')

## 4. Inspect drill scoring on a single URL

Edit `inspect_url` below to dig into why a particular drill picked the URL it did. This calls `find_primary_url` directly — same code path as the pipeline.

In [132]:
inspect_url   = 'https://www.ajc.com/accessatl/2026/05/the-ultimate-weekend-checklist-10-metro-atlanta-events-you-absolutely-need-to-check-out/'
inspect_title = '20th Annual Taste of East Cobb 2026'

primary = find_primary_url(inspect_url, title=inspect_title)
print('Drill result:', primary or '(no primary chosen)')

      ↳ drilled 'https://www.ajc.com/accessatl/2026/05/the-ultimate-weekend-c…' → 'https://www.awesomealpharetta.com/taste-of-alpharetta' (anchor='awesomealpharetta.com', heading='Taste of Alpharetta', score=10)
Drill result: https://www.awesomealpharetta.com/taste-of-alpharetta


## 4b. Expand a listicle into individual event candidates

Instead of dropping a listicle, scrape it and pull out every plausible event URL inside. Each becomes its own candidate (with title from the surrounding heading where possible).

In [138]:
from aggregator_drilldown import expand_listicle

listicle_url = 'https://www.mdjonline.com/news/frontpage/out-and-about-5-things-to-do-this-weekend-in-cobb-county-may-15-/article_200a61d3-c605-4e51-bb96-356d410d1f32.html'
events = expand_listicle(listicle_url)
print(f'Extracted {len(events)} event candidates from {listicle_url}\n')
for i, ev in enumerate(events, 1):
    print(f"[{i:2}] {ev['title']}")
    print(f"     url:    {ev['url']}")
    if ev.get('summary'):
        print(f"     anchor: {ev['summary']}")
    print()

Extracted 14 event candidates from https://www.mdjonline.com/news/frontpage/out-and-about-5-things-to-do-this-weekend-in-cobb-county-may-15-/article_200a61d3-c605-4e51-bb96-356d410d1f32.html

[ 1] Art of the Cocktail
     url:    http://mariettacobbartmuseum.org
     anchor: Madeline Beck, curator for the Marietta-Cobb Museum of Art, examines a wooden bowl piece at the museum’s “Georgia Wood Artists” juried exhibition. Proceeds from the upcoming Art of the Cocktail event benefit the museum. from the Marietta Daily Journal Madeline Beck, curator for the Marietta-Cobb Museum of Art, examines a wooden bowl piece at the museum’s “Georgia Wood Artists” juried exhibition. Proceeds from the upcoming Art of the Cocktail event benefit the museum. Madeline Beck, curator for the Marietta-Cobb Museum of Art, examines a wooden bowl piece at the museum’s “Georgia Wood Artists” juried exhibition. Proceeds from the upcoming Art of the Cocktail event benefit the museum. from the Marietta Daily Journal 

## 5. Stage 3 — Date filter

After listicle drop + drill, the date filter checks title + summary + article body + primary-text for a date on or after the upcoming-Friday floor. Anything purely about past dates gets removed.

In [ ]:
# Use the pool built in the previous cell (skip the legacy drill_down logic).
# Date filter now scans title + summary + article_text + primary_text +
# listicle_date_hint + url + source_url — same as Featured_Event.py.

floor = upcoming_friday()
print(f'Date floor: {floor}  (drop anything dated < this)')
print()

kept, past_urls = filter_candidates_by_date(
    pool, floor,
    text_keys=('title', 'summary', 'article_text', 'primary_text',
               'listicle_date_hint', 'url', 'source_url'),
)
kept_urls = {c['url'] for c in kept}

def _emit(c, status):
    print(f"[{status}]  host={_hostname(c.get('url',''))}")
    print(f"  title: {c.get('title','')}")
    print(f"  url:   {c.get('url','')}")
    dates = c.get('dates_found') or []
    if dates:
        print(f"  dates: {', '.join(dates)}")
    print()

print(f'=== KEPT ({len(kept)} candidates) — will reach Claude ===')
for c in pool:
    if c.get('url') in kept_urls:
        _emit(c, 'KEPT')

print(f'=== DROPPED ({len(pool) - len(kept)} candidates) — past-only ===')
for c in pool:
    if c.get('url') not in kept_urls:
        _emit(c, 'DROPPED (past-only)')

## 6. Final pool (what Claude sees)

In [ ]:
print(f'Final pool: {len(kept)} candidates\n')
for i, c in enumerate(kept, 1):
    print(f'{i}. {c.get("title","?")[:80]}')
    print(f'   {c.get("url","")}')
    print()